In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import shutil
import glob
import logging
import xml.etree.ElementTree as ET

from pathlib import Path
from io import StringIO
from typing import Dict, List, Optional, Union
from functools import reduce
import multiprocess as mp
from multiprocessing import cpu_count

import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from tqdm.contrib.logging import logging_redirect_tqdm
import logging

import parmed
from openbabel import openbabel

from rdkit import Chem
from rdkit.Chem import rdmolops, AllChem, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

from plip.structure.preparation import PDBComplex, logger as PLIP_LOGGER
from plip.exchange.report import StructureReport
from plip.basic import config as PLIP_CONFIG

PLIP_LOGGER.setLevel(logging.ERROR)
PLIP_LOGGER.propagate = False

# Collect Interactions

In [2]:
cwdir = os.getcwd()
dataset_dir = '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset'
subdirs = ['biolip2_opt/',
           'pdbbind_opt_poly/',
           'pdbbind_opt_sm/']

def generate_PDBIDs():
    fcsv = os.path.join(dataset_dir, 'PDBIDs.csv')
    df_PDBIDs = pd.read_csv(fcsv)
    PDBIDs = (df_PDBIDs.T).to_numpy()
    PDBIDs = PDBIDs.tolist()
    PDBIDs = [[x for x in row if x is not None and not (isinstance(x, float) and math.isnan(x))]for row in PDBIDs]
    return PDBIDs

def generate_args():
    args = []
    for index, subdir in enumerate(subdirs):
        for PDBID in PDBIDs[index]:
            args.append((
                PDBID,
                subdir,
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_ligand_refined.sdf'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_ligand_refined.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_protein_refined.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_complex.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/interaction.xml')
            ))
    return args

PDBIDs = generate_PDBIDs()
args = generate_args()

In [3]:
def extract_pi_cation(xmlpath):
    xmlobj = ET.parse(xmlpath)
    binding_sites = xmlobj.findall("./bindingsite")
    # print(binding_sites)
    bs = [bs for bs in binding_sites if bs.findall("identifiers/longname")[0].text == "UNL"][0]
    pications = bs.findall("interactions/pi_cation_interactions/pi_cation_interaction")
    infos = []
    for pication in pications:
        info = {}
        for attr in pication:
            if attr.tag == 'ligcoo' or attr.tag == 'protcoo':
                for s in ['x', 'y', 'z']:
                    info[f'{attr.tag}_{s}'] = attr.find(s).text
            elif attr.tag == 'lig_idx_list' or attr.tag == 'prot_idx_list':
                info[attr.tag] = ' '.join([idx.text for idx in attr.findall('idx')])
            else:
                info[attr.tag] = attr.text
        infos.append(info)
    return infos

def make_identifier(code, resnr, restype, lig_idx_list, prot_idx_list):
    lig_idx_list = lig_idx_list.split()
    prot_idx_list = prot_idx_list.split()
    
    if len(lig_idx_list) > 1:
        lig_idx_list = f'{lig_idx_list[0]}-{lig_idx_list[len(lig_idx_list)-1]}'
    else:
        lig_idx_list = lig_idx_list[0]
    
    if len(prot_idx_list) > 1:
        prot_idx_list = f'{prot_idx_list[0]}-{prot_idx_list[len(prot_idx_list)-1]}'
    else:
        prot_idx_list = prot_idx_list[0]
    
    return f'{code}_{resnr}_{restype}_{lig_idx_list}_{prot_idx_list}'

def make_identifier_from_row(row):
    return make_identifier(row['PDBID'], row['resnr'], row['restype'], row['prot_idx_list'], row['lig_idx_list'])

In [4]:
df = []
err = []

for arg in tqdm(args):
    try:
        infos = extract_pi_cation(arg[6])
        for info in infos:
            info['PDBID'] = arg[0][:4]
            info['full_PDBID'] = arg[0]
            info['subdir'] = arg[1]
            df.append(info)
    except Exception as e:
        err.append(arg)

print(len(err))
# print(LigNumberErr)

df = pd.DataFrame(df)
cols = ['PDBID'] + ['full_PDBID'] + ['subdir'] + list(df.columns[:-3])
df = df[cols]

df.index = [make_identifier_from_row(row) for index, row in df.iterrows()]
fcsv = os.path.join(cwdir, 'pi_cation.csv')
df.to_csv(fcsv)
df

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18928/18928 [00:07<00:00, 2566.74it/s]


0


,PDBID,full_PDBID,subdir,resnr,restype,reschain,prot_idx_list,resnr_lig,restype_lig,reschain_lig,...,offset,protcharged,lig_group,lig_idx_list,ligcoo_x,ligcoo_y,ligcoo_z,protcoo_x,protcoo_y,protcoo_z
1fkx_212_HIS_3347-3353_1-6,1fkx,1fkx_PRH_A_353,biolip2_opt/,212,HIS,A,3347 3353,1,UNL,A,...,1.23,True,Aromatic,1 2 3 4 5 6,2.729,20.216,38.678,4.111,24.463,39.679
1iss_350_LYS_5540_4-9,1iss,1iss_MCG_B_2001,biolip2_opt/,350,LYS,B,5540,1,UNL,Z,...,1.95,True,Aromatic,4 5 6 7 8 9,53.527,39.371,107.246,48.603,41.233,107.435
1knr_348_HIS_5375-5381_31-38,1knr,1knr_FAD_A_800,biolip2_opt/,348,HIS,A,5375 5381,1,UNL,Z,...,1.40,True,Aromatic,31 32 33 35 37 38,61.224,45.214,134.230,60.043,49.096,134.793
1maa_84_TRP_1340-1346_12,1maa,1maa_DME_A_998,biolip2_opt/,84,TRP,A,1340 1341 1343 1344 1346,1,UNL,Z,...,1.83,False,quartamine,12,81.512,110.149,49.987,85.839,110.587,51.451
1maa_84_TRP_1343-1353_12,1maa,1maa_DME_A_998,biolip2_opt/,84,TRP,A,1343 1346 1347 1349 1351 1353,1,UNL,Z,...,0.98,False,quartamine,12,81.512,110.149,49.987,85.533,111.726,49.648
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6szp_167_HIS_2564-2570_10-12,6szp,6szp_M3B_A_300,pdbbind_opt_sm/,167,HIS,A,2564 2565 2566 2568 2570,1,UNL,Z,...,0.59,False,guanidine,10 11 12,-3.524,21.885,2.897,-4.893,23.302,-0.860
6uf0_93_ARG_1418-1424_15-36,6uf0,6uf0_Q5V_A_701,pdbbind_opt_sm/,93,ARG,A,1418 1421 1424,1,UNL,Z,...,1.76,True,Aromatic,15 16 17 18 35 36,-12.384,-25.782,-13.744,-14.652,-21.462,-14.909
6v5l_6_LYS_117_7-20,6v5l,6v5l_QPD_A_203,pdbbind_opt_sm/,6,LYS,A,117,1,UNL,Z,...,1.77,True,Aromatic,7 8 10 19 20,9.825,4.347,-6.363,13.727,4.299,-4.449
6v5l_6_LYS_117_1-20,6v5l,6v5l_QPD_A_203,pdbbind_opt_sm/,6,LYS,A,117,1,UNL,Z,...,1.40,True,Aromatic,1 6 11 14 19 20,9.921,2.665,-5.138,13.727,4.299,-4.449


# Data Cleaning
## Lysine Error
Plip doesn't process insertion code for resnr numbers

In [5]:
lys_atom = []
lys_atoms = []

for index, row in df.iterrows():
    if row['restype'] == 'LYS':
        if len(row['prot_idx_list'].split()) == 1:
            lys_atom.append(row['full_PDBID'])
        else: 
            lys_atoms.append([row['full_PDBID'], row['subdir'], len(row['prot_idx_list'].split())])

print(len(lys_atoms))
print(len(lys_atom))
lys_atoms

0
516


[]

## Remove His Cation
Histidine cations are not really cations ($pKa \approx 6$, around $10 \%$ total His protonated in physiological pH), can't characterize with stationary time frame 

In [6]:
hisCation = []

for index, row in df.iterrows():
    if row['restype'] == 'HIS' and row['lig_group'] == "Aromatic":
        hisCation.append([row['full_PDBID'], row['subdir']])

print(len(hisCation))
# hisCation

239


In [7]:
df = df[~((df['restype'] == 'HIS') & (df['lig_group'] == "Aromatic"))]

len(df)
# df

2041

## Merge Duplicated Ring Interactions

Combine rows if protein or ligand has attatched ring system to prevent double counting

In [8]:
df.to_csv('pi_cation_unmerged.csv')

grouped = df.groupby(['full_PDBID', 'resnr'])

In [9]:
def merge_duplicate(group):
    """
    Merge duplicate rows in a group

    Parameters
    ----------
    group: pandas.core.groupby.DataFrameGroupBy
        DataFrameGroupBy object grouped by ['full_PDBID', 'resnr']
    
    Return
    ------
    result_rows: pandas.DataFrame
        DataFrame of merged rows
    """

    if len(group) == 1:  # Return if interaction is unique
        return [group]

    group = group.copy()
    group['lig_idx_set'] = group['lig_idx_list'].apply(lambda x: set(map(int, str(x).split())))
    group['prot_idx_set'] = group['prot_idx_list'].apply(lambda x: set(map(int, str(x).split())))

    rows = group.to_dict('records')  # Convert to list of dicts
    adjacency_list = {i: set() for i in range(len(rows))}  # Graph representation

    # Build adjacency list for connected components
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            if rows[i]['lig_idx_set'].intersection(rows[j]['lig_idx_set']):  # Overlapping ligands
                adjacency_list[i].add(j)
                adjacency_list[j].add(i)

    # Find connected components using DFS (Depth-First Search)
    visited = set()
    merged_groups = []

    def dfs(node, component): # visits every node connection through recursion, builds list of connected nodes (component)
        if node in visited:
            return
        visited.add(node) # keeps history of visited nodes
        component.append(node) # appending connected nodes
        for neighbor in adjacency_list[node]: # for loop through every connection
            dfs(neighbor, component)

    for i in range(len(rows)):
        if i not in visited:
            component = []
            dfs(i, component)
            merged_groups.append(component) # add list of connected nodes (component) to merged_groups

    result_rows = []
    for component in merged_groups: # Merge each connected component
        merged_row = rows[component[0]].copy()
        for idx in component[1:]:
            merged_row['lig_idx_set'].update(rows[idx]['lig_idx_set'])
            merged_row['lig_idx_list'] = ' '.join(map(str, sorted(merged_row['lig_idx_set'])))
            merged_row['prot_idx_set'].update(rows[idx]['prot_idx_set'])
            merged_row['prot_idx_list'] = ' '.join(map(str, sorted(merged_row['prot_idx_set'])))
        
        result_rows.append({k: v for k, v in merged_row.items() if k not in ['lig_idx_set', 'prot_idx_set']})

    return [pd.DataFrame(result_rows)]

In [10]:
edge_cases = pd.DataFrame() # residues with 2+ seperate ligand group interactions (based on lig_idx_list)

def return_group(group):
    return group

for _, group in grouped: 
    if len(group) > 2:
        edge_cases = pd.concat([edge_cases, return_group(group)], ignore_index=True)

edge_cases

,PDBID,full_PDBID,subdir,resnr,restype,reschain,prot_idx_list,resnr_lig,restype_lig,reschain_lig,...,offset,protcharged,lig_group,lig_idx_list,ligcoo_x,ligcoo_y,ligcoo_z,protcoo_x,protcoo_y,protcoo_z
0,2xzg,2xzg_VH1_A_1362,pdbbind_opt_sm/,66,ARG,A,1083 1086 1089,1,UNL,Z,...,1.78,True,Aromatic,1 2 3 5 6 7,46.444,-6.486,25.432,51.384,-8.975,26.105
1,2xzg,2xzg_VH1_A_1362,pdbbind_opt_sm/,66,ARG,A,1083 1086 1089,1,UNL,Z,...,1.30,True,Aromatic,9 10 12 13 25 26,48.616,-10.745,24.325,51.384,-8.975,26.105
2,2xzg,2xzg_VH1_A_1362,pdbbind_opt_sm/,66,ARG,A,1083 1086 1089,1,UNL,Z,...,1.56,True,Aromatic,13 21 22 23 24 25,49.968,-12.463,25.400,51.384,-8.975,26.105
3,2xzg,2xzg_VH1_A_1362,pdbbind_opt_sm/,66,ARG,A,1083 1086 1089,1,UNL,Z,...,1.37,True,Aromatic,12 13 14 15 20 21,50.705,-11.394,23.323,51.384,-8.975,26.105
4,4qnb,4qnb_1B0_A_301,pdbbind_opt_sm/,71,LYS,A,1126,1,UNL,Z,...,1.12,True,Aromatic,23 24 29 30 31,15.315,26.377,3.562,18.056,27.335,1.877
5,4qnb,4qnb_1B0_A_301,pdbbind_opt_sm/,71,LYS,A,1126,1,UNL,Z,...,1.53,True,Aromatic,24 25 26 27 28 29,15.560,28.472,4.069,18.056,27.335,1.877
6,4qnb,4qnb_1B0_A_301,pdbbind_opt_sm/,71,LYS,A,1126,1,UNL,Z,...,0.22,True,Aromatic,16 17 18 19 20 21,18.089,22.078,0.113,18.056,27.335,1.877
7,4u0e,4u0e_1B0_A_302,pdbbind_opt_sm/,71,LYS,A,1126,1,UNL,Z,...,1.85,True,Aromatic,24 25 26 27 28 29,-17.087,-27.605,-4.686,-14.413,-29.077,-2.505
8,4u0e,4u0e_1B0_A_302,pdbbind_opt_sm/,71,LYS,A,1126,1,UNL,Z,...,0.95,True,Aromatic,23 24 29 30 31,-15.356,-26.351,-4.279,-14.413,-29.077,-2.505
9,4u0e,4u0e_1B0_A_302,pdbbind_opt_sm/,71,LYS,A,1126,1,UNL,Z,...,0.38,True,Aromatic,16 17 18 19 20 21,-10.178,-26.721,-0.823,-14.413,-29.077,-2.505


In [11]:
result_rows = []

for _, group in grouped: 
    result_rows.extend(merge_duplicate(group))

df = pd.concat(result_rows, ignore_index=True)
df.index = [make_identifier_from_row(row) for index, row in df.iterrows()]
fcsv = os.path.join(cwdir, 'pi_cation.csv')
df.to_csv(fcsv)
df

,PDBID,full_PDBID,subdir,resnr,restype,reschain,prot_idx_list,resnr_lig,restype_lig,reschain_lig,...,offset,protcharged,lig_group,lig_idx_list,ligcoo_x,ligcoo_y,ligcoo_z,protcoo_x,protcoo_y,protcoo_z
1a08_15_ARG_311-314_9-14,1a08,1a08_ACE-DIP_C_100-103,pdbbind_opt_poly/,15,ARG,A,311 313 314,1,UNL,Z,...,1.19,True,Aromatic,9 10 11 12 13 14,45.105,8.445,22.911,43.146,10.021,27.458
1a1b_15_ARG_309-312_9-14,1a1b,1a1b_ACE-DIP_C_100-103,pdbbind_opt_poly/,15,ARG,A,309 311 312,1,UNL,Z,...,1.28,True,Aromatic,9 10 11 12 13 14,45.902,8.644,23.082,43.653,10.018,27.433
1a1c_15_ARG_304-307_9-14,1a1c,1a1c_ACE-DIX_C_100-103,pdbbind_opt_poly/,15,ARG,A,304 306 307,1,UNL,Z,...,1.28,True,Aromatic,9 10 11 12 13 14,45.288,8.503,23.538,42.920,10.002,27.969
1a1c_63_LYS_1072_9-14,1a1c,1a1c_ACE-DIX_C_100-103,pdbbind_opt_poly/,63,LYS,A,1072,1,UNL,Z,...,1.89,True,Aromatic,9 10 11 12 13 14,45.288,8.503,23.538,48.025,5.760,21.863
1a1e_14_ARG_293-296_9-14,1a1e,1a1e_ACE-DIY_C_100-103,pdbbind_opt_poly/,14,ARG,A,293 295 296,1,UNL,Z,...,1.57,True,Aromatic,9 10 11 12 13 14,45.221,8.633,23.306,42.960,10.181,27.760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7mxn_382_LYS_6228_1-25,7mxn,7mxn_ZR1_A_701,biolip2_opt/,382,LYS,A,6228,1,UNL,Z,...,1.17,True,Aromatic,1 3 5 12 25,-35.508,32.352,-3.964,-36.167,28.987,0.409
7ng7_40_LYS_702_22-29,7ng7,7ng7_UCW_A_602,biolip2_opt/,40,LYS,A,702,1,UNL,Z,...,1.95,True,Aromatic,22 23 24 27 28 29,-15.597,-3.957,-5.862,-12.479,-6.072,-7.232
7upj_108_ARG_1770-1776_29-34,7upj,7upj_INU_A_100,pdbbind_opt_sm/,108,ARG,B,1770 1773 1776,1,UNL,Z,...,1.49,True,Aromatic,29 30 31 32 33 34,7.447,27.812,3.066,6.151,30.800,0.801
7vkn_178_ARG_2820-2826_8-14,7vkn,7vkn_7GI_A_801,biolip2_opt/,178,ARG,A,2820 2823 2826,1,UNL,Z,...,1.41,True,Aromatic,8 9 10 11 13 14,-19.047,-44.794,-15.201,-18.108,-48.714,-16.701


## Tyrosine Cation-Pi, H-Bonding Check

In [12]:
def extract_hbond(xmlpath):
    xmlobj = ET.parse(xmlpath)
    binding_sites = xmlobj.findall("./bindingsite")
    bs = [bs for bs in binding_sites if bs.findall("identifiers/longname")[0].text == "MOL"][0]
    hbonds = bs.findall("interactions/hydrogen_bonds/hydrogen_bond")
    infos = []
    for hbond in hbonds:
        info = {}
        for attr in hbond:
            if attr.tag == 'ligcoo' or attr.tag == 'protcoo':
                for s in ['x', 'y', 'z']:
                    info[f'{attr.tag}_{s}'] = attr.find(s).text
            else:
                info[attr.tag] = attr.text
        infos.append(info)
    return infos

In [13]:
TYR = df[(df['restype'] == 'TYR') & (df['lig_group'] != "Aromatic")]
print(len(TYR))
# TYR

215


## Tertamine Protonation
kick unprotonated tertamine cations

In [16]:
rows = df[df['lig_group'] == 'tertamine'].index.tolist()
tertamine_idx = list(map(int, df[df['lig_group'] == 'tertamine']['lig_idx_list'].tolist()))

tertamine_args = []

for arg in pications:
    if any(arg[8] in row for row in rows):
        tertamine_args.append(arg)


results = []

for index, arg in tqdm(enumerate(tertamine_args), total=len(tertamine_args)):
    mol = Chem.MolFromPDBFile(arg[3], removeHs=False, sanitize=False)
    try:
        atom = mol.GetAtomWithIdx(tertamine_idx[index] - 1)
        
        if atom.GetSymbol() != 'N':
            result = {'error': 'Not a nitrogen atom'}
            continue

        neighbors = atom.GetNeighbors()
        h_count = sum(1 for nei in neighbors if nei.GetSymbol() == 'H')
        heavy_count = sum(1 for nei in neighbors if nei.GetSymbol() != 'H')
        formal_charge = atom.GetFormalCharge()        
        is_protonated = formal_charge > 0 or (heavy_count == 3 and h_count >= 1)
        
        result = {
            'index': arg[8],
            'is_protonated': is_protonated,
            'formal_charge': formal_charge,
            'hydrogen_count': h_count,
            'heavy_neighbor_count': heavy_count,
            'total_neighbors': len(neighbors)
        }
        
    except Exception as e:
        result = {'error': str(e)}

    results.append(result)

results = pd.DataFrame(results)
results

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 165/165 [00:03<00:00, 45.87it/s]


,index,is_protonated,formal_charge,hydrogen_count,heavy_neighbor_count,total_neighbors
0,1d3d_79_TRP_1308-1321_33,True,1,1,3,4
1,1d3p_79_TRP_1311-1324_33,True,1,1,3,4
2,1d3q_79_TRP_1311-1324_33,True,1,1,3,4
3,1dg9_132_TYR_2095-2104_4,False,0,0,3,3
4,1eve_330_PHE_5125-5134_14,True,1,1,3,4
...,...,...,...,...,...,...
160,6u34_84_TRP_1307-1320_7,True,1,1,3,4
161,6u9k_72_TYR_1243-1252_44,True,1,1,3,4
162,6u9m_71_TYR_1212-1221_26,True,1,1,3,4
163,6u9r_72_TYR_1227-1236_37,True,1,1,3,4


In [17]:
not_prot = results[results['is_protonated'] == False]
not_prot_index = not_prot['index'].tolist()
print(len(not_prot_index))

df = df.drop(index=not_prot_index)
fcsv = os.path.join(cwdir, 'pi_cation.csv')
df.to_csv(fcsv)
df

23


,PDBID,full_PDBID,subdir,resnr,restype,reschain,prot_idx_list,resnr_lig,restype_lig,reschain_lig,...,offset,protcharged,lig_group,lig_idx_list,ligcoo_x,ligcoo_y,ligcoo_z,protcoo_x,protcoo_y,protcoo_z
1a08_15_ARG_311-314_9-14,1a08,1a08_ACE-DIP_C_100-103,pdbbind_opt_poly/,15,ARG,A,311 313 314,1,UNL,Z,...,1.19,True,Aromatic,9 10 11 12 13 14,45.105,8.445,22.911,43.146,10.021,27.458
1a1b_15_ARG_309-312_9-14,1a1b,1a1b_ACE-DIP_C_100-103,pdbbind_opt_poly/,15,ARG,A,309 311 312,1,UNL,Z,...,1.28,True,Aromatic,9 10 11 12 13 14,45.902,8.644,23.082,43.653,10.018,27.433
1a1c_15_ARG_304-307_9-14,1a1c,1a1c_ACE-DIX_C_100-103,pdbbind_opt_poly/,15,ARG,A,304 306 307,1,UNL,Z,...,1.28,True,Aromatic,9 10 11 12 13 14,45.288,8.503,23.538,42.920,10.002,27.969
1a1c_63_LYS_1072_9-14,1a1c,1a1c_ACE-DIX_C_100-103,pdbbind_opt_poly/,63,LYS,A,1072,1,UNL,Z,...,1.89,True,Aromatic,9 10 11 12 13 14,45.288,8.503,23.538,48.025,5.760,21.863
1a1e_14_ARG_293-296_9-14,1a1e,1a1e_ACE-DIY_C_100-103,pdbbind_opt_poly/,14,ARG,A,293 295 296,1,UNL,Z,...,1.57,True,Aromatic,9 10 11 12 13 14,45.221,8.633,23.306,42.960,10.181,27.760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7mxn_382_LYS_6228_1-25,7mxn,7mxn_ZR1_A_701,biolip2_opt/,382,LYS,A,6228,1,UNL,Z,...,1.17,True,Aromatic,1 3 5 12 25,-35.508,32.352,-3.964,-36.167,28.987,0.409
7ng7_40_LYS_702_22-29,7ng7,7ng7_UCW_A_602,biolip2_opt/,40,LYS,A,702,1,UNL,Z,...,1.95,True,Aromatic,22 23 24 27 28 29,-15.597,-3.957,-5.862,-12.479,-6.072,-7.232
7upj_108_ARG_1770-1776_29-34,7upj,7upj_INU_A_100,pdbbind_opt_sm/,108,ARG,B,1770 1773 1776,1,UNL,Z,...,1.49,True,Aromatic,29 30 31 32 33 34,7.447,27.812,3.066,6.151,30.800,0.801
7vkn_178_ARG_2820-2826_8-14,7vkn,7vkn_7GI_A_801,biolip2_opt/,178,ARG,A,2820 2823 2826,1,UNL,Z,...,1.41,True,Aromatic,8 9 10 11 13 14,-19.047,-44.794,-15.201,-18.108,-48.714,-16.701


# Fragment Protein

In [15]:
df = pd.read_csv("pi_cation.csv", index_col=0)

pications = []

for index, row in tqdm(df.iterrows()):
    pications.append((
        row['PDBID'], 
        row['full_PDBID'],
        row['subdir'],
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/{row['full_PDBID']}_complex.pdb"),
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/interaction.xml"),
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/PiCatProteinFragment"),
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/PiCatLigandFragment"), 
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/{row['full_PDBID']}_ligand_refined.sdf"),
        index, 
        row['resnr'], 
        row['restype'],
        row['reschain'],
        row['lig_idx_list'],
    ))

1839it [00:00, 16805.00it/s]


In [18]:
FRAG_SIDE_CHAIN = {
    'LYS': ['NZ', 'HZ1', 'HZ2', 'HZ3', 'CE', 'HE2', 'HE3'],
    'ARG': ['NH1', 'HH11', 'HH12', 'NH2', 'HH21', 'HH22', 'CZ', 'NE', 'HE1', 'CD', 'HD2', 'HD3'],
    'PHE': ['CZ', 'CE1', 'HE1', 'CE2', 'HE2', 'CD1', 'HD1', 'CD2', 'HD2', 'CG'],
    'TYR': ['OH', 'HH', 'CZ', 'CE1', 'HE1', 'CE2', 'HE2', 'CD1', 'HD1', 'CD2', 'HD2', 'CG'],
    'TRP': ['CH2', 'HH2', 'CZ2', 'HZ2', 'CZ3', 'HZ3', 'NE1', 'HE1', 'CE2', 'CE3', 'HE3', 'CD1', 'HD1', 'CD2', 'CG'],
    'HIS': ['CE1', 'HE1', 'NE2', 'HE2', 'ND1', 'HD1', 'CD2', 'HD2', 'CG'],
    'HIE': ['CE1', 'HE1', 'NE2', 'HE2', 'ND1', 'CD2', 'HD2', 'CG'],
    'HID': ['CE1', 'HE1', 'NE2', 'ND1', 'HD1', 'CD2', 'HD2', 'CG'],
}


def get_residue_fragment_mask(
    struct, resnr, restype, reschain
):
    residues = struct[f':{resnr}'].residues
    assert len(residues) == 1, "More than one residues are selected"
    assert residues[0].name == restype, f"Selected residue is not {restype}"
    assert residues[0].chain == reschain, f"Selected residue does not in chain {reschain}"

    resnr = int(resnr)
    return f':{resnr}@{",".join(FRAG_SIDE_CHAIN[restype])}'


def struct_to_pdb_string(struct):
    strio = StringIO()
    struct.save(strio, format='pdb')
    strio.seek(0)
    return strio.read()


def add_hydrogen(struct, output_pdb):
    obConversion = openbabel.OBConversion()
    obConversion.SetInAndOutFormats("pdb", "pdb")
    
    ob_mol = openbabel.OBMol()
    obConversion.ReadString(ob_mol, struct_to_pdb_string(struct))
    
    resnames = set()
    atnames = []
    for atom in openbabel.OBMolAtomIter(ob_mol):
        residue = atom.GetResidue()
        resname = residue.GetName()
        atname = residue.GetAtomID(atom).strip()
        atnames.append(atname)
        resnames.add(resname)
    resnames = list(resnames)
        
    ob_mol.AddHydrogens()
    obConversion.WriteFile(ob_mol, output_pdb)

In [19]:
warnings.filterwarnings("ignore")

def wrapper_prot_frag(pications):
    struct = parmed.load_file(pications[3])
    # infos = extract_pi_cation(pications[4])
    
    errs = []
    # for interaction in infos:
    try:
        mask = get_residue_fragment_mask(struct, pications[9], pications[10], pications[11])
        frag_path = os.path.join(pications[5], f'{pications[8]}.pdb')
        add_hydrogen(struct[mask], frag_path)
    except Exception as e:
        errs.append((pications[1], pications[2], interaction, str(e)))
    return errs
    


if __name__ == '__main__':
    
    reset = True
    for arg in tqdm(pications):
        if reset and os.path.isdir(arg[5]):
            shutil.rmtree(arg[5])
            
        if not os.path.isdir(arg[5]):
            os.mkdir(arg[5])

    errs = []
    with mp.Pool(cpu_count()) as pool:
        errs = list(tqdm(pool.imap(wrapper_prot_frag, pications), total=len(pications)))
    
    for err in errs:
        if len(err) > 0:
            print(err)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1839/1839 [02:01<00:00, 15.08it/s]


# Fragment Ligands

In [20]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from tqdm import tqdm
import sys
sys.path.insert(0, '../')
from edautils.fragment import get_fg_set, select_fragment, write_sdf

In [21]:
frags = []
for arg in tqdm(pications):
    mol = Chem.SDMolSupplier(arg[7], removeHs=False, sanitize=True)[0]
    # idx = mol.GetAtomWithIdx(int(arg[8]) - 1).GetNeighbors()[0].GetIdx()
    idx_list = [int(idx) - 1 for idx in arg[12].split()]
    fgs = get_fg_set(mol, False)

    target_atoms = set()
    for idx in idx_list:
        if idx in fgs:
            target_atoms.update(fgs[idx])
    
    frag = select_fragment(mol, fgs[idx])
    frag.SetProp("identifier", arg[8])

    if not os.path.isdir(arg[6]):
        os.mkdir(arg[6])
    write_sdf(frag, os.path.join(arg[6], f'{arg[8]}.sdf'))

    frag_no_h = Chem.RemoveHs(frag)
    AllChem.Compute2DCoords(frag_no_h)
    frags.append(frag_no_h)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1839/1839 [00:12<00:00, 146.44it/s]


# Collect To Output

In [22]:
from edautils.fragment import read_pdb, extract_info

In [23]:
errPDBID = ['2rfn_AM7_B_1'] # nitrogens in ARG group connected

In [24]:
valid_prot_smiles = {
    'C[NH3+]', # LYS
    'CNC(N)=[NH2+]', # ARG
    'C1=CC=CC=C1', # PHE
    'OC1=CC=CC=C1', # TYR
    'C1=CC=C2NC=CC2=C1', # TRP
    'C1=CC2=C(C=C1)NC=C2', # TRP
    'C1=CNC=N1' # HIS
}

for index, row in tqdm(df.iterrows(), total=len(df)):
    PDBID = row['full_PDBID']
    if PDBID in errPDBID:
        print(PDBID)
        df.drop(index, inplace=True)
    else:
        prot_frag_dir = os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/PiCatProteinFragment")
        lig_frag_dir = os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/PiCatLigandFragment")
            
        prot_frag_file = os.path.join(prot_frag_dir, f'{index}.pdb')
        lig_frag_file = os.path.join(lig_frag_dir, f'{index}.sdf')
    
        lig_frag = Chem.SDMolSupplier(lig_frag_file, removeHs=False)[0]
        prot_frag = read_pdb(prot_frag_file)
    
        lig_info = extract_info(lig_frag)
        prot_info = extract_info(prot_frag)
        
        assert prot_info['smiles'] in valid_prot_smiles, f'Invalid protin fragment: {prot_info["smiles"]}'
    
        df.loc[index, 'natoms0'] = lig_info['natoms']
        df.loc[index, 'charge0'] = lig_info['charge']
        df.loc[index, 'smiles0'] = lig_info['smiles']
        df.loc[index, 'natoms1'] = prot_info['natoms']
        df.loc[index, 'charge1'] = prot_info['charge']
        df.loc[index, 'smiles1'] = prot_info['smiles']
        df.loc[index, 'elements'] = ' '.join(lig_info['elements'] + prot_info['elements'])
        df.loc[index, 'xyz'] = ' '.join([f'{x:.3f}' for x in lig_info['xyz'].flatten()] + \
            [f'{x:.3f}' for x in prot_info['xyz'].flatten()])

df_cc = df.query('(`charge0` == 0 & `charge1` == 1) | (`charge0` == 1 & `charge1` == 0)')

df_cc.to_csv('pi_cation.csv')
df_cc

 29%|███████████████████████████████████████████████████████████▏                                                                                                                                               | 529/1816 [00:00<00:00, 1345.66it/s]

2rfn_AM7_B_1


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1816/1816 [00:01<00:00, 1392.27it/s]


,PDBID,full_PDBID,subdir,resnr,restype,reschain,prot_idx_list,resnr_lig,restype_lig,reschain_lig,...,protcoo_y,protcoo_z,natoms0,charge0,smiles0,natoms1,charge1,smiles1,elements,xyz
1aj6_66_ARG_1040-1046_20-28,1aj6,1aj6_NOV_A_1,pdbbind_opt_sm/,66,ARG,A,1040 1043 1046,1,UNL,Z,...,-11.157,37.114,39.0,0.0,CO[C@H](C)OC1=C(C)C2=C(C=C1)C(O)=C(NC(C)=O)C(=...,13.0,1.0,CNC(N)=[NH2+],C C O C O C C C C C C C O C O C C O N C O C H ...,57.181 -4.381 37.605 57.198 -5.074 36.249 57.7...
1ao0_767_ARG_11624-11630_18-27,1ao0,1ao0_ADP_A_468,pdbbind_opt_sm/,767,ARG,B,11624 11627 11630,1,UNL,Z,...,4.828,74.682,18.0,0.0,CN1C=NC2=C1N=CN=C2N,13.0,1.0,CNC(N)=[NH2+],C N C N C C N N C N C H H H H H H H C H H N C ...,12.185 4.839 77.078 12.870 3.631 76.666 14.020...
1b0h_405_ARG_6533-6539_18-24,1b0h,1b0h_LYS-LYS_B_601-603,pdbbind_opt_poly/,405,ARG,A,6533 6536 6539,1,UNL,A,...,23.933,21.784,21.0,0.0,CC1=CC=CC2=C1C=CC=C2,13.0,1.0,CNC(N)=[NH2+],C C C C C C C C C C C H H H H H H H H H H C H ...,3.924 21.666 16.842 5.097 21.790 17.804 6.173 ...
1bhf_14_ARG_306-312_7-12,1bhf,1bhf_ACE-ILE_I_251-255,pdbbind_opt_poly/,14,ARG,A,306 309 312,1,UNL,A,...,-4.915,-1.391,18.0,0.0,CC1=CC=C(C)C=C1,13.0,1.0,CNC(N)=[NH2+],C C C C C C C C H H H H H H H H H H C H H N C ...,27.302 -0.274 -1.273 28.774 -0.282 -1.527 29.6...
1bhf_62_LYS_1048_7-12,1bhf,1bhf_ACE-ILE_I_251-255,pdbbind_opt_poly/,62,LYS,A,1048,1,UNL,A,...,3.299,-3.619,18.0,0.0,CC1=CC=C(C)C=C1,8.0,1.0,C[NH3+],C C C C C C C C H H H H H H H H H H C H H N H ...,27.302 -0.274 -1.273 28.774 -0.282 -1.527 29.6...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7mxn_382_LYS_6228_1-25,7mxn,7mxn_ZR1_A_701,biolip2_opt/,382,LYS,A,6228,1,UNL,Z,...,28.987,0.409,20.0,0.0,CC1=C2C=CN(C)C2=NC=N1,8.0,1.0,C[NH3+],C C C C C C C C N N N H H H H H H H H H C H H ...,-36.526 31.891 -4.388 -35.307 31.301 -4.422 -3...
7ng7_40_LYS_702_22-29,7ng7,7ng7_UCW_A_602,biolip2_opt/,40,LYS,A,702,1,UNL,Z,...,-6.072,-7.232,35.0,0.0,COC(=O)NC1=C(OC)C=C(C2=NNC3=NC=NC=C23)C=C1,8.0,1.0,C[NH3+],C C C N N N N C C C C C O C C C C N C O O C H ...,-19.966 -4.211 -6.334 -19.105 -4.941 -7.099 -1...
7upj_108_ARG_1770-1776_29-34,7upj,7upj_INU_A_100,pdbbind_opt_sm/,108,ARG,B,1770 1773 1776,1,UNL,Z,...,30.800,0.801,20.0,0.0,CNS(=O)(=O)C1=CC=CC=C1,13.0,1.0,CNC(N)=[NH2+],C N S O O C C C C C C H H H H H H H H H C H H ...,9.280 26.926 6.950 7.891 27.277 6.908 7.503 28...
7vkn_178_ARG_2820-2826_8-14,7vkn,7vkn_7GI_A_801,biolip2_opt/,178,ARG,A,2820 2823 2826,1,UNL,Z,...,-48.714,-16.701,19.0,0.0,COC1=CC=C(F)C=C1C,13.0,1.0,CNC(N)=[NH2+],C O C C C C F C C C H H H H H H H H H C H H N ...,-17.766 -46.551 -12.167 -17.363 -46.028 -13.42...


In [27]:
errs = []

for arg in tqdm(pications):
    try:
        if arg[8] not in df_cc.index:
            shutil.rmtree(arg[5])
            shutil.rmtree(arg[6])
    except Exception as e:
        errs.append((arg[8], e))

errs

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1839/1839 [00:00<00:00, 14801.52it/s]


[('1a1c_63_LYS_1072_9-14', FileNotFoundError(2, 'No such file or directory')),
 ('1a1e_62_LYS_1061_9-14', FileNotFoundError(2, 'No such file or directory')),
 ('1ex8_54_TYR_850-859_36-39',
  FileNotFoundError(2, 'No such file or directory')),
 ('1jyq_55_LYS_928_17-22', FileNotFoundError(2, 'No such file or directory')),
 ('3chg_212_TRP_3311-3324_5',
  FileNotFoundError(2, 'No such file or directory')),
 ('3chg_59_TRP_907-920_5', FileNotFoundError(2, 'No such file or directory')),
 ('3mks_366_ARG_5953-5959_3-10',
  FileNotFoundError(2, 'No such file or directory')),
 ('3qbc_287_PHE_4597-4606_5-7',
  FileNotFoundError(2, 'No such file or directory')),
 ('3ud5_54_TYR_857-866_20-24',
  FileNotFoundError(2, 'No such file or directory')),
 ('3udv_54_TYR_863-872_21-25',
  FileNotFoundError(2, 'No such file or directory')),
 ('4crj_56_PHE_860-869_13-19',
  FileNotFoundError(2, 'No such file or directory')),
 ('4cwb_56_PHE_866-875_24-23',
  FileNotFoundError(2, 'No such file or directory')),
 (

# Tinkering

In [83]:
restype = set()
lig_group = set()

for index, row in df.iterrows():
    restype.add(row['restype'])
    lig_group.add(row['lig_group'])

print(f'restypes: {restype}')
print(f'lig_groups: {lig_group}\n')

restype_lig_group = {
    'LYS': set(), 
    'TRP': set(), 
    'TYR': set(), 
    'PHE': set(), 
    'HIS': set(), 
    'ARG': set()
}

for index, row in df.iterrows():
    restype_lig_group[row['restype']].add(row['lig_group'])

cation_atom_length = {
    'guanidine': set(), 
    'quartamine': set(), 
    'sulfonium': set(), 
    'tertamine': set(), 
    'LYS': set(), 
    'ARG': set()
}

for index, row in df.iterrows():
    if row['restype'] == 'LYS' or row['restype'] == 'ARG':
        cation_atom_length[row['restype']].add(len(row['prot_idx_list'].split()))
    elif row['restype'] == 'HIS':
        continue
    else:
        cation_atom_length[row['lig_group']].add(len(row['lig_idx_list'].split()))

print(f'restype_ligand interaction pairs: {restype_lig_group}\n')
print(f'cation_atom_length: {cation_atom_length}')

restypes: {'LYS', 'ARG', 'HIS', 'TRP', 'PHE', 'TYR'}
lig_groups: {'quartamine', 'tertamine', 'sulfonium', 'Aromatic', 'guanidine'}

restype_ligand interaction pairs: {'LYS': {'Aromatic'}, 'TRP': {'quartamine', 'tertamine', 'sulfonium', 'guanidine'}, 'TYR': {'quartamine', 'tertamine', 'sulfonium', 'guanidine'}, 'PHE': {'quartamine', 'sulfonium', 'tertamine', 'guanidine'}, 'HIS': {'tertamine', 'guanidine'}, 'ARG': {'Aromatic'}}

cation_atom_length: {'guanidine': {3}, 'quartamine': {1}, 'sulfonium': {1}, 'tertamine': {1}, 'LYS': {1}, 'ARG': {3}}
